In [ ]:
from data_labeling_pipeline.grandtour_missions import get_mission_by_index

downloaded_missions = [0, 16, 18, 30]
mission_index = 30
print(f"Mission index: {mission_index}")
mission = get_mission_by_index(mission_index)

In [ ]:
from pathlib import Path

import cv2
import numpy as np
from data_labeling_pipeline.dataset_loader import GrandTourLoader
from tqdm.auto import tqdm

# Inputs
mission_path = Path(f"/datahome/lmertens/grand_tour_dataset/{mission}")
labels_dir = Path(f"./output/{mission}")
images_dir = mission_path / "images" / "zed2i_right_images"

# Output video settings
video_path = labels_dir / "traversability_with_right_images.mp4"
fps = 10.0
max_frames = 1500
stride = 1
resize_factor = 0.3

# Load labels and trajectories via loader
label_ts = np.load(labels_dir / "timestamp.npy")
trav_labels = np.load(labels_dir / "trav_labels.npy")
if trav_labels.ndim == 1:
    trav_labels = trav_labels[:, None]

# Load sensor data via loader with correct frame convention
loader = GrandTourLoader(mission_path)
shared_ts, U, Z = loader.load_data_shared_timestamps(dt=0.1, fill_gaps=True)
image_ts = loader.load_subtopic("zed2i_right_images", "timestamp")

# Extract commands and odometry
v_cmd, omega_cmd = U.T
px, py, yaw = Z.T

# Compute actual velocities in body frame
vx = np.gradient(px, shared_ts)
vy = np.gradient(py, shared_ts)
v_act = np.cos(yaw) * vx + np.sin(yaw) * vy  # Forward body velocity
omega_act = np.gradient(yaw, shared_ts)

# Interpolate to label timestamps
v_cmd_at_label = np.interp(label_ts, shared_ts, v_cmd)
omega_cmd_at_label = np.interp(label_ts, shared_ts, omega_cmd)
v_act_at_label = np.interp(label_ts, shared_ts, v_act)
omega_act_at_label = np.interp(label_ts, shared_ts, omega_act)
dv = v_cmd_at_label - v_act_at_label
d_omega = omega_cmd_at_label - omega_act_at_label

# Map labels to nearest image frames
insert_idx = np.searchsorted(image_ts, label_ts)
insert_idx = np.clip(insert_idx, 1, len(image_ts) - 1)
left_idx = insert_idx - 1
right_idx = insert_idx
nearest_idx = np.where(
    np.abs(label_ts - image_ts[left_idx]) <= np.abs(image_ts[right_idx] - label_ts),
    left_idx,
    right_idx,
)

# Select frame IDs for video
frame_ids = np.where(nearest_idx >= 0)[0]
if stride > 1:
    frame_ids = frame_ids[::stride]
if max_frames is not None:
    frame_ids = frame_ids[:max_frames]
if len(frame_ids) == 0:
    raise ValueError("No valid aligned frames found.")

# Normalize label bars using percentiles
low = np.percentile(trav_labels, 1, axis=0)
high = np.percentile(trav_labels, 99, axis=0)
scale = np.maximum(high - low, 1e-8)


def to_unit(values: np.ndarray) -> np.ndarray:
    return np.clip((values - low) / scale, 0.0, 1.0)


# Infer output dimensions from first frame
first_image_idx = int(nearest_idx[frame_ids[0]])
first_img_path = images_dir / f"{first_image_idx:06d}.jpeg"
first_img = cv2.imread(str(first_img_path), cv2.IMREAD_COLOR)
if first_img is None:
    raise FileNotFoundError(f"Could not read first image: {first_img_path}")

if resize_factor != 1.0:
    first_img = cv2.resize(
        first_img,
        None,
        fx=resize_factor,
        fy=resize_factor,
        interpolation=cv2.INTER_AREA,
    )

img_h, img_w = first_img.shape[:2]
panel_h = 200
frame_h = img_h + panel_h

# Initialize video writer
writer = cv2.VideoWriter(
    str(video_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (img_w, frame_h),
)
if not writer.isOpened():
    raise RuntimeError("Failed to open video writer.")

# Render frames
bar_colors = [(0, 200, 0), (0, 170, 255), (220, 120, 0), (200, 0, 160)]
label_count = trav_labels.shape[1]

for k in tqdm(frame_ids, desc="Writing video frames"):
    image_idx = int(nearest_idx[k])
    img = cv2.imread(str(images_dir / f"{image_idx:06d}.jpeg"), cv2.IMREAD_COLOR)
    if img is None:
        continue

    if resize_factor != 1.0:
        img = cv2.resize(
            img, None, fx=resize_factor, fy=resize_factor, interpolation=cv2.INTER_AREA
        )
    if img.shape[:2] != (img_h, img_w):
        img = cv2.resize(img, (img_w, img_h), interpolation=cv2.INTER_AREA)

    # Create overlay panel
    panel = np.full((panel_h, img_w, 3), 245, dtype=np.uint8)

    # Text: linear velocities
    cv2.putText(
        panel,
        f"v_cmd: {v_cmd_at_label[k]: .3f} m/s  |  v_act: {v_act_at_label[k]: .3f} m/s  |  dv: {dv[k]: .3f}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.56,
        (35, 35, 35),
        1,
        cv2.LINE_AA,
    )

    # Text: angular velocities
    cv2.putText(
        panel,
        f"omega_cmd: {omega_cmd_at_label[k]: .3f} rad/s  |  omega_act: {omega_act_at_label[k]: .3f} rad/s",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (35, 35, 35),
        1,
        cv2.LINE_AA,
    )

    # Text: omega difference on its own line
    cv2.putText(
        panel,
        f"d_omega: {d_omega[k]: .3f} rad/s",
        (20, 88),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (35, 35, 35),
        1,
        cv2.LINE_AA,
    )

    # Draw traversability bars
    values = trav_labels[k]
    unit_values = to_unit(values)
    base_y = panel_h - 30
    max_bar_h = panel_h - 100
    spacing = img_w // max(label_count, 1)
    bar_w = max(24, min(100, spacing // 3))

    for j in range(label_count):
        center_x = int((j + 0.5) * spacing)
        x0 = center_x - bar_w // 2
        x1 = center_x + bar_w // 2
        bar_h = int(unit_values[j] * max_bar_h)
        y0 = base_y - bar_h
        color = bar_colors[j % len(bar_colors)]

        cv2.rectangle(panel, (x0, y0), (x1, base_y), color, thickness=-1)
        cv2.rectangle(panel, (x0, base_y - max_bar_h), (x1, base_y), (80, 80, 80), 1)
        cv2.putText(
            panel,
            f"m[{j}]={values[j]:.3f}",
            (max(5, x0 - 8), panel_h - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.46,
            (30, 30, 30),
            1,
            cv2.LINE_AA,
        )

    writer.write(np.vstack([img, panel]))

writer.release()
print(f"Saved: {video_path}")